# EDA Visual & Spasial untuk Menentukan Resolusi Target

Notebook ini menganalisis dataset gambar untuk menentukan resolusi target yang efisien secara saintifik.

Fokus analisis:
- pemetaan dimensi gambar $(W, H)$
- distribusi rasio aspek $AR = W/H$
- statistik persentil untuk kandidat resolusi target
- K-Means dengan $K=1$ sebagai kompromi resolusi
- analisis relative area untuk anotasi polygon/mask
- evaluasi strategi resize, crop, dan padding

In [ ]:
from __future__ import annotations

import json
import math
import os
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 9

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "camo" / "environment_annotation"
TARGET_LABEL = "camou"
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
TARGET_MULTIPLE = 32

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATASET_DIR  = {DATASET_DIR}")


def find_image_path(annotation_path: Path) -> Path | None:
    for ext in IMAGE_EXTENSIONS:
        candidate = annotation_path.with_suffix(ext)
        if candidate.exists():
            return candidate
    return None


def read_image_metadata(image_path: Path) -> tuple[int, int, str]:
    """Read metadata without loading the full pixel array into RAM."""
    with Image.open(image_path) as image:
        width, height = image.size
        fmt = image.format or image_path.suffix.replace(".", "").upper()
    return int(width), int(height), str(fmt)


def round_up_to_multiple(value: float, multiple: int = TARGET_MULTIPLE) -> int:
    return int(math.ceil(float(value) / float(multiple)) * multiple)


def labelme_relative_area(annotation_path: Path, width: int, height: int, target_label: str = TARGET_LABEL) -> float:
    """Compute relative target area from LabelMe polygon annotations."""
    with annotation_path.open("r", encoding="utf-8") as f:
        annotation = json.load(f)

    mask = np.zeros((height, width), dtype=np.uint8)
    found_target = False

    for shape in annotation.get("shapes", []):
        if shape.get("label") != target_label:
            continue
        points = np.asarray(shape.get("points", []), dtype=np.float32)
        if points.shape[0] < 3:
            continue
        points[:, 0] = np.clip(np.round(points[:, 0]), 0, width - 1)
        points[:, 1] = np.clip(np.round(points[:, 1]), 0, height - 1)
        polygon = points.astype(np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(mask, [polygon], 1)
        found_target = True

    if not found_target:
        return 0.0
    return float(mask.mean())


In [ ]:
annotation_paths = sorted(DATASET_DIR.glob("*.json"))
records = []
missing_pairs = []

for annotation_path in annotation_paths:
    image_path = find_image_path(annotation_path)
    if image_path is None:
        missing_pairs.append(annotation_path.name)
        continue

    width, height, image_format = read_image_metadata(image_path)
    aspect_ratio = width / height if height else float("nan")
    orientation = "square"
    if width > height:
        orientation = "landscape"
    elif height > width:
        orientation = "portrait"

    records.append(
        {
            "filename": image_path.name,
            "annotation": annotation_path.name,
            "format": image_format,
            "width": width,
            "height": height,
            "aspect_ratio": aspect_ratio,
            "orientation": orientation,
            "relative_area": labelme_relative_area(annotation_path, width, height, TARGET_LABEL),
        }
    )

if pd is not None and records:
    df = pd.DataFrame(records)
else:
    df = None

print(f"Jumlah anotasi terdeteksi     : {len(annotation_paths)}")
print(f"Pasangan image-annotation     : {len(records)}")
print(f"Anotasi tanpa pasangan image  : {len(missing_pairs)}")
if missing_pairs:
    print("Contoh file tanpa pasangan:")
    for name in missing_pairs[:5]:
        print(f"  - {name}")

if records:
    sample_preview = records[0]
    print("\nContoh record pertama:")
    for key in ["filename", "width", "height", "aspect_ratio", "orientation", "relative_area"]:
        print(f"  {key:>14}: {sample_preview[key]}")


In [ ]:
if not records:
    raise RuntimeError(f"Tidak ada pasangan image-annotation yang ditemukan di {DATASET_DIR}")

widths = np.array([r["width"] for r in records], dtype=np.float32)
heights = np.array([r["height"] for r in records], dtype=np.float32)
aspect_ratios = np.array([r["aspect_ratio"] for r in records], dtype=np.float32)
relative_areas = np.array([r["relative_area"] for r in records], dtype=np.float32)
orientations = np.array([r["orientation"] for r in records], dtype=object)
filenames = [r["filename"] for r in records]

orientation_counts = Counter(orientations.tolist())
print("Distribusi orientasi:")
for key in ["landscape", "portrait", "square"]:
    print(f"  {key:>10}: {orientation_counts.get(key, 0)}")

color_map = {
    "landscape": "tab:blue",
    "portrait": "tab:orange",
    "square": "tab:green",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for orient in ["landscape", "portrait", "square"]:
    mask = orientations == orient
    if np.any(mask):
        ax.scatter(
            widths[mask],
            heights[mask],
            s=26,
            alpha=0.75,
            label=f"{orient} ({int(mask.sum())})",
            color=color_map[orient],
            edgecolors="none",
        )

max_dim = int(max(widths.max(), heights.max()))
ax.plot([0, max_dim], [0, max_dim], linestyle="--", color="black", linewidth=1, label="1:1 line")
ax.set_title("Scatter Plot Lebar vs Tinggi")
ax.set_xlabel("Lebar (W)")
ax.set_ylabel("Tinggi (H)")
ax.set_xlim(0, max_dim * 1.05)
ax.set_ylim(0, max_dim * 1.05)
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

# Annotasi outlier berdasarkan penyimpangan aspect ratio dari square
outlier_scores = np.abs(np.log2(aspect_ratios + 1e-8))
outlier_indices = np.argsort(outlier_scores)[-5:]
for idx in outlier_indices:
    ax.annotate(
        filenames[idx],
        (widths[idx], heights[idx]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=7,
        alpha=0.8,
    )

ax = axes[1]
ax.hist(aspect_ratios, bins=30, color="tab:purple", alpha=0.8, edgecolor="white")
for ratio, label, color in [(1.0, "1:1", "black"), (4 / 3, "4:3", "tab:blue"), (16 / 9, "16:9", "tab:red")]:
    ax.axvline(ratio, linestyle="--", color=color, linewidth=1.5, label=label)
ax.set_title("Histogram Rasio Aspek")
ax.set_xlabel("Aspect Ratio (W/H)")
ax.set_ylabel("Jumlah gambar")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def percentile_summary(values: np.ndarray) -> dict[str, float]:
    return {
        "min": float(np.min(values)),
        "p50": float(np.percentile(values, 50)),
        "p75": float(np.percentile(values, 75)),
        "max": float(np.max(values)),
        "mean": float(np.mean(values)),
    }

width_summary = percentile_summary(widths)
height_summary = percentile_summary(heights)
aspect_summary = percentile_summary(aspect_ratios)

candidate_square_50 = round_up_to_multiple(max(width_summary["p50"], height_summary["p50"]), TARGET_MULTIPLE)
candidate_square_75 = round_up_to_multiple(max(width_summary["p75"], height_summary["p75"]), TARGET_MULTIPLE)

coverage_50 = float(np.mean((widths <= candidate_square_50) & (heights <= candidate_square_50)))
coverage_75 = float(np.mean((widths <= candidate_square_75) & (heights <= candidate_square_75)))

print("Ringkasan ukuran gambar")
print(f"  Lebar  : min={width_summary['min']:.0f} | median={width_summary['p50']:.0f} | p75={width_summary['p75']:.0f} | max={width_summary['max']:.0f}")
print(f"  Tinggi : min={height_summary['min']:.0f} | median={height_summary['p50']:.0f} | p75={height_summary['p75']:.0f} | max={height_summary['max']:.0f}")
print(f"  AR     : min={aspect_summary['min']:.3f} | median={aspect_summary['p50']:.3f} | p75={aspect_summary['p75']:.3f} | max={aspect_summary['max']:.3f}")
print()
print("Kandidat resolusi target berbasis persentil")
print(f"  Square dari median   : {candidate_square_50} x {candidate_square_50} (cakupan downscale-only ≈ {coverage_50 * 100:.1f}%)")
print(f"  Square dari p75      : {candidate_square_75} x {candidate_square_75} (cakupan downscale-only ≈ {coverage_75 * 100:.1f}%)")

samples = np.column_stack([widths, heights]).astype(np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
compactness, labels, centers = cv2.kmeans(samples, 1, None, criteria, 10, cv2.KMEANS_PP_CENTERS)
center_w, center_h = centers[0]
kmeans_square = round_up_to_multiple(max(center_w, center_h), TARGET_MULTIPLE)

print()
print("K-Means K=1")
print(f"  Centroid W/H  : ({center_w:.1f}, {center_h:.1f})")
print(f"  Square hasil  : {kmeans_square} x {kmeans_square}")
print(f"  Compactness   : {compactness:.2f}")

In [ ]:
relative_area_percent = relative_areas * 100.0
small_threshold = 5.0
small_mask = relative_area_percent < small_threshold
small_count = int(np.sum(small_mask))
small_fraction = float(np.mean(small_mask))

print("Analisis relative area target")
print(f"  Median relative area   : {np.median(relative_area_percent):.3f}%")
print(f"  p75 relative area      : {np.percentile(relative_area_percent, 75):.3f}%")
print(f"  Min / Max relative area: {relative_area_percent.min():.3f}% / {relative_area_percent.max():.3f}%")
print(f"  Gambar di bawah 5%     : {small_count} / {len(relative_area_percent)} ({small_fraction * 100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(relative_area_percent, bins=30, color="tab:teal", alpha=0.8, edgecolor="white")
ax.axvline(small_threshold, linestyle="--", color="red", linewidth=1.5, label="5% threshold")
ax.set_title("Histogram Relative Area Target")
ax.set_xlabel("Relative area (%)")
ax.set_ylabel("Jumlah gambar")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(widths, relative_area_percent, s=24, alpha=0.7, color="tab:olive")
ax.set_title("Relative Area vs Lebar Gambar")
ax.set_xlabel("Lebar (W)")
ax.set_ylabel("Relative area (%)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
aspect_iqr = float(np.percentile(aspect_ratios, 75) - np.percentile(aspect_ratios, 25))
median_ar = float(np.median(aspect_ratios))
small_target_rate = float(np.mean(relative_area_percent < 5.0))

if aspect_iqr < 0.15:
    aspect_message = "Rasio aspek relatif rapat; resize ke bentuk square masih masuk akal jika target resolusi tidak terlalu kecil."
else:
    aspect_message = "Rasio aspek menyebar; zero-padding atau random cropping biasanya lebih aman daripada memaksa resize langsung ke square."

if small_target_rate >= 0.25:
    target_message = "Banyak target berukuran kecil; hindari downscaling agresif karena detail mikro berisiko hilang."
else:
    target_message = "Ukuran target cukup aman; resolusi menengah-besar masih layak dipertimbangkan."

print("Rekomendasi praktis")
print(f"  Median AR               : {median_ar:.3f}")
print(f"  IQR AR                  : {aspect_iqr:.3f}")
print(f"  Rasio target < 5%       : {small_target_rate * 100:.1f}%")
print()
print(f"  {aspect_message}")
print(f"  {target_message}")
print()
print("  Ringkasan strategi:")
print("   - Resize langsung: cocok jika aspect ratio rapat dan target relatif besar.")
print("   - Zero-padding   : cocok jika input model harus square tetapi Anda ingin mempertahankan rasio asli.")
print("   - Random crop    : cocok untuk generasi tekstur karena menjaga ketajaman lokal dari gambar resolusi tinggi.")
print(f"  Kandidat resolusi square awal: median={candidate_square_50}x{candidate_square_50} | p75={candidate_square_75}x{candidate_square_75} | kmeans={kmeans_square}x{kmeans_square}")

## Evaluasi Strategi Resize, Crop, dan Padding

### Resize langsung
- Paling sederhana untuk pipeline training.
- Berisiko merusak detail jika rasio aspek menyebar lebar atau jika objek target sangat kecil.

### Zero-padding
- Mempertahankan rasio aspek asli.
- Aman secara struktural, tetapi menambahkan area kosong yang tidak membawa informasi.

### Random cropping
- Disarankan bila gambar asli beresolusi tinggi dan Anda ingin menjaga ketajaman tekstur lokal.
- Cocok untuk tugas generasi tekstur / camouflage karena model tetap melihat detail dunia nyata tanpa interpolasi berlebihan.

### Cara membaca hasil notebook ini
- Jika scatter W vs H rapat di sekitar garis 1:1, square resize lebih aman.
- Jika histogram AR menumpuk di 4:3, 16:9, atau rasio non-square lainnya, padding/cropping biasanya lebih masuk akal.
- Jika relative area mask banyak di bawah 5%, hindari downscaling agresif karena detail target dapat hilang.